# Incident Ops OpenEnv Training Notebook

This Colab notebook runs the full hackathon pipeline:

1. Install dependencies
2. Clone the repo
3. Smoke-test the environment and reward functions
4. Run a short sanity GRPO pass
5. Run the full SFT + GRPO training pipeline
6. Save `reward_curve.png`
7. Compare the base model against the trained model

Recommended runtime: `T4 GPU`.


In [ ]:
import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    raise RuntimeError('No GPU found. In Colab, switch to Runtime -> Change runtime type -> T4 GPU.')


In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -r https://raw.githubusercontent.com/Christy-saji/incident-ops-openenv/master/requirements-train.txt
!pip install -q pandas matplotlib httpx
print('Dependencies installed')


In [ ]:
import os
import sys

REPO_URL = 'https://github.com/Christy-saji/incident-ops-openenv'
REPO_DIR = '/content/incident-ops-openenv'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!git log --oneline -5


In [ ]:
from env.environment import DevOpsEnv
from graders.grader import compute_score
from tasks.task_config import TASK_CONFIGS
from train import (
    format_reward_func,
    step_reward_func,
    anti_cheat_reward_func,
    task_alignment_reward_func,
    sequence_progress_reward_func,
)
import json

print('Environment smoke test')
for task in TASK_CONFIGS:
    env = DevOpsEnv(task=task)
    obs = env.reset()
    _, reward, _, _ = env.step('acknowledge_incident')
    score, _ = compute_score(task, env._state)
    print(f'  {task:12s} reward={reward:+.2f} score={score:.2f} phase={obs["incident_phase"]}')

partial_env = DevOpsEnv(task='hard', partial_obs=True)
partial_obs = partial_env.reset()
assert partial_obs['known_findings'][0].startswith('[hidden')

prompt = [[
    {'role': 'system', 'content': 'sys'},
    {'role': 'user', 'content': json.dumps({'task': 'easy', 'recent_actions': []})},
]]
completion = [[{'content': 'inspect_deploy_history'}]]
print('format_reward_func:', format_reward_func(prompt, completion))
print('step_reward_func:', step_reward_func(prompt, completion))
print('anti_cheat_reward_func:', anti_cheat_reward_func(prompt, completion))
print('task_alignment_reward_func:', task_alignment_reward_func(prompt, completion))
print('sequence_progress_reward_func:', sequence_progress_reward_func(prompt, completion))
print('Smoke tests passed')


In [ ]:
import gc
import os
import torch
from unsloth import FastLanguageModel, PatchDPOTrainer
from trl import GRPOTrainer, GRPOConfig
from train import (
    format_reward_func,
    step_reward_func,
    anti_cheat_reward_func,
    task_alignment_reward_func,
    generate_prompts,
    RewardLoggerCallback,
)

PatchDPOTrainer()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-1B-Instruct',
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    use_gradient_checkpointing='unsloth',
)

dataset = generate_prompts(per_task_n=2, mid_episode_n=2)
sanity_args = GRPOConfig(
    output_dir='outputs_sanity',
    learning_rate=1e-5,
    max_steps=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    logging_steps=1,
    num_generations=2,
    max_prompt_length=256,
    max_completion_length=16,
)
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward_func,
        step_reward_func,
        anti_cheat_reward_func,
        task_alignment_reward_func,
    ],
    args=sanity_args,
    train_dataset=dataset,
    callbacks=[RewardLoggerCallback(log_path='outputs_sanity/reward_log.csv')],
)
trainer.train()
assert os.path.exists('outputs_sanity/reward_log.csv')
print('Sanity GRPO run finished')

del model, tokenizer, trainer
gc.collect()
torch.cuda.empty_cache()


In [ ]:
import os

os.environ['GRPO_MAX_STEPS'] = '300'
os.environ['GRPO_PER_TASK_PROMPTS'] = '8'
os.environ['GRPO_MID_EPISODE_PROMPTS'] = '60'

%run train.py


In [ ]:
import os
import pandas as pd
from IPython.display import Image, display

assert os.path.exists('outputs_grpo/reward_log.csv')
assert os.path.exists('reward_curve.png')

df = pd.read_csv('outputs_grpo/reward_log.csv')
print(df.tail(10).to_string())
display(Image(filename='reward_curve.png'))


In [ ]:
!python compare_inference.py --base-model unsloth/Llama-3.2-1B-Instruct --trained-model ./trained_sre_agent


In [ ]:
import os

HF_TOKEN = ''
HF_REPO_ID = 'hf-username/model-repo-name'

if HF_TOKEN:
    from huggingface_hub import login
    from unsloth import FastLanguageModel

    login(token=HF_TOKEN)
    push_model, push_tokenizer = FastLanguageModel.from_pretrained(
        model_name='./trained_sre_agent',
        max_seq_length=1024,
        dtype=None,
        load_in_4bit=False,
    )
    push_model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    push_tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print(f'Uploaded to https://huggingface.co/{HF_REPO_ID}')
else:
    print('Set HF_TOKEN if you want to publish the trained checkpoint.')


In [ ]:
from google.colab import files

files.download('reward_curve.png')
files.download('outputs_grpo/reward_log.csv')
